# Configuration-search setup

This notebook creates a reproducible catalog of variants on Google Drive for preliminary TRAIN/DEV experiments:

- extraction prompt variants;
- schemas with different numbers of relations;
- a separate YAML configuration for every combination;
- document-pool and evaluation-sample size presets;
- one `experiment_registry.json` consumed by the runner notebook.

It makes no LLM calls and does not use TEST. After editing the definitions, you can run this notebook again: the registry and configurations will be updated deterministically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Paths and common settings

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/DKR-KG-Construction/KG-construction.git'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content/KG-construction')
DRIVE_ROOT = Path('/content/drive/MyDrive/DKR_KG_Construction')
EXPERIMENT_ROOT = DRIVE_ROOT / 'configuration_search_full96'

for directory in (
    EXPERIMENT_ROOT / 'prompts',
    EXPERIMENT_ROOT / 'schemas',
    EXPERIMENT_ROOT / 'configs',
    EXPERIMENT_ROOT / 'raw',
    EXPERIMENT_ROOT / 'runs',
    EXPERIMENT_ROOT / 'comparisons',
):
    directory.mkdir(parents=True, exist_ok=True)

print('Experiment root:', EXPERIMENT_ROOT)

## 2. Repository, dependencies, and Re-DocRED

In [ ]:
import os
import subprocess
import sys

def run(command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)

if not (REPO_DIR / '.git').exists():
    run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(REPO_DIR)])
else:
    run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH])
    run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH])
    run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH])

run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements.lock')])
run(['bash', 'run.sh', 'data'], cwd=REPO_DIR)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')

## 3. Editable experiment variants

This is the main configuration cell. By default, it prepares relation-count variants up to all 96 Re-DocRED relations found in TRAIN. Entity types are intentionally kept fixed: changing the type set would change the task itself. Relation names and descriptions are cached from Wikidata, while the 14 tracked project descriptions take precedence where available.

In [ ]:
import copy
import json
import yaml

BASELINE_PROMPT_NAME = 'baseline'
BASELINE_RELATION_COUNT = 14
RELATION_COUNTS = [10, 14, 20, 40, 60, 96]
PAIR_POLICY = 'observed'  # observed | project_thresholds

# pool_size is the number of documents actually sent to the LLM.
# eval_sizes lists the nested sample sizes evaluated within that pool.
SIZE_PRESETS = {
    'pilot':  {'pool_size': 10,  'eval_sizes': [10],               'seeds': [0]},
    'small':  {'pool_size': 50,  'eval_sizes': [25],               'seeds': [0, 1, 2, 3, 4]},
    'medium': {'pool_size': 200, 'eval_sizes': [50, 100],          'seeds': [0, 1, 2, 3, 4]},
    'large':  {'pool_size': 500, 'eval_sizes': [200],              'seeds': [0, 1, 2, 3, 4]},
    'all':    {'pool_size': 500, 'eval_sizes': [25, 50, 100, 200], 'seeds': [0, 1, 2, 3, 4]},
}

base_prompt = (REPO_DIR / 'prompts' / 'extract_v1.txt').read_text(encoding='utf-8').strip() + '\n'

def with_extra_rules(base, title, rules):
    block = title + '\n' + '\n'.join(f'- {rule}' for rule in rules) + '\n\n'
    marker = 'Output format'
    if marker not in base:
        raise ValueError(f'Marker {marker!r} not found in base prompt')
    return base.replace(marker, block + marker, 1)

PROMPT_VARIANTS = {
    'baseline': {
        'description': 'Tracked extract_v1 prompt without changes.',
        'text': base_prompt,
    },
    'evidence_strict': {
        'description': 'Requires explicit endpoint and relation evidence in the cited sentence.',
        'text': with_extra_rules(base_prompt, 'Additional evidence rules', [
            'Create a triple only when the cited sentence identifies both endpoints by a listed mention or an unambiguous pronoun.',
            'The quotation must contain the words that express the relation, not merely mention the two entities.',
            'Do not infer citizenship, origin, membership, location, or notability from background knowledge.',
        ]),
    },
    'conservative': {
        'description': 'Precision-oriented prompt that prefers omission when evidence is ambiguous.',
        'text': with_extra_rules(base_prompt, 'Additional conservative rules', [
            'When a relation is ambiguous, omit it; an empty triples list is preferable to a speculative fact.',
            'Do not convert co-occurrence, list membership, authorship context, or topical association into a relation unless the relation definition is directly satisfied.',
            'Before returning each triple, verify head-to-tail direction and entity types against the relation description.',
        ]),
    },
}

print('Prompt variants:', list(PROMPT_VARIANTS))
print('Relation counts:', RELATION_COUNTS)
print('Size presets:', SIZE_PRESETS)

## 4. Generate prompts, schemas, configurations, and the registry

Relations are discovered and ranked by frequency directly from `train_revised.json`. Every smaller variant is therefore a deterministic subset of every larger one. The registry records the exact IDs, metadata source, type-pair policy, and share of TRAIN facts covered by each relation set. The cached full catalog is built without using DEV or TEST labels.

In [ ]:
from collections import Counter, defaultdict
from datetime import datetime, timezone
import requests

base_config_path = REPO_DIR / 'config.yaml'
tracked_schema_path = REPO_DIR / 'schema' / 'schema.json'
train_path = REPO_DIR / 'data' / 'Re-DocRED' / 'train_revised.json'
base_cfg = yaml.safe_load(base_config_path.read_text(encoding='utf-8'))
tracked_schema = json.loads(tracked_schema_path.read_text(encoding='utf-8'))
train_docs = json.loads(train_path.read_text(encoding='utf-8'))
tracked_specs = {item['id']: item for item in base_cfg['schema']['relations']}

relation_counts = Counter()
pair_counts = defaultdict(Counter)
entity_type_counts = Counter()
for doc in train_docs:
    vertex_set = doc['vertexSet']
    for entity in vertex_set:
        for mention in entity:
            entity_type_counts[mention['type']] += 1
    for label in doc['labels']:
        relation_id = label['r']
        head_type = vertex_set[label['h']][0]['type']
        tail_type = vertex_set[label['t']][0]['type']
        relation_counts[relation_id] += 1
        pair_counts[relation_id][(head_type, tail_type)] += 1

ranked_relation_ids = sorted(relation_counts, key=lambda rid: (-relation_counts[rid], rid))
full_relation_ids = list(ranked_relation_ids)
total_train_facts = sum(relation_counts.values())
print(f'Discovered {len(full_relation_ids)} positive relation types in TRAIN')

if BASELINE_RELATION_COUNT not in RELATION_COUNTS:
    raise ValueError('BASELINE_RELATION_COUNT must be included in RELATION_COUNTS')
if not RELATION_COUNTS or min(RELATION_COUNTS) < 1:
    raise ValueError('RELATION_COUNTS must contain positive integers')
if max(RELATION_COUNTS) > len(full_relation_ids):
    raise ValueError(
        f'Requested up to {max(RELATION_COUNTS)} relations, but TRAIN contains only '
        f'{len(full_relation_ids)} positive relation types.'
    )
if BASELINE_PROMPT_NAME not in PROMPT_VARIANTS:
    raise ValueError('BASELINE_PROMPT_NAME is missing from PROMPT_VARIANTS')
if PAIR_POLICY not in {'observed', 'project_thresholds'}:
    raise ValueError("PAIR_POLICY must be 'observed' or 'project_thresholds'")

metadata_path = EXPERIMENT_ROOT / 'relation_metadata_wikidata_en.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8')) if metadata_path.exists() else {}
missing_metadata = [rid for rid in full_relation_ids if rid not in metadata]
session = requests.Session()
session.headers.update({'User-Agent': 'DKR-KG-Construction/0.1 academic experiment'})
for start in range(0, len(missing_metadata), 50):
    batch = missing_metadata[start:start + 50]
    response = session.get(
        'https://www.wikidata.org/w/api.php',
        params={
            'action': 'wbgetentities', 'format': 'json', 'formatversion': 2,
            'ids': '|'.join(batch), 'props': 'labels|descriptions', 'languages': 'en',
        },
        timeout=60,
    )
    response.raise_for_status()
    response_entities = response.json().get('entities', {})
    if isinstance(response_entities, dict):
        response_entities = response_entities.values()
    for entity in response_entities:
        rid = entity['id']
        metadata[rid] = {
            'name': entity.get('labels', {}).get('en', {}).get('value', rid),
            'description': entity.get('descriptions', {}).get('en', {}).get(
                'value', f'Wikidata property {rid}'),
            'source': 'wikidata_api_en',
        }
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

still_missing = [rid for rid in full_relation_ids if rid not in metadata]
if still_missing:
    raise RuntimeError(f'Missing Wikidata metadata for: {still_missing}')

relation_specs = {}
for rid in full_relation_ids:
    if rid in tracked_specs:
        relation_specs[rid] = copy.deepcopy(tracked_specs[rid])
        relation_specs[rid]['metadata_source'] = 'tracked_project_description'
    else:
        relation_specs[rid] = {
            'id': rid, 'name': metadata[rid]['name'],
            'description': metadata[rid]['description'],
            'metadata_source': metadata[rid]['source'],
        }

min_pair_count = int(base_cfg['schema']['min_pair_count'])
min_pair_share = float(base_cfg['schema']['min_pair_share'])
full_schema = {
    'entity_types': sorted(entity_type_counts),
    'relations': {},
    'built_from': {
        'n_train_docs': len(train_docs), 'relation_count': len(full_relation_ids),
        'relation_ranking': 'descending_train_count_then_relation_id',
        'pair_policy': PAIR_POLICY, 'min_pair_count': min_pair_count,
        'min_pair_share': min_pair_share, 'metadata_cache': str(metadata_path),
    },
}
for rid in full_relation_ids:
    total = relation_counts[rid]
    if PAIR_POLICY == 'observed':
        allowed_pairs = [list(pair) for pair, _ in pair_counts[rid].most_common()]
    else:
        allowed_pairs = [
            list(pair) for pair, count in pair_counts[rid].most_common()
            if count >= min_pair_count and count / total >= min_pair_share
        ]
    full_schema['relations'][rid] = {
        'name': relation_specs[rid]['name'],
        'description': relation_specs[rid]['description'],
        'metadata_source': relation_specs[rid]['metadata_source'],
        'train_count': total, 'allowed_pairs': allowed_pairs,
    }

full_catalog_path = EXPERIMENT_ROOT / 'schemas' / 'all_train_relations.json'
full_catalog_path.write_text(
    json.dumps(full_schema, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

prompt_paths = {}
for name, spec in PROMPT_VARIANTS.items():
    path = EXPERIMENT_ROOT / 'prompts' / f'{name}.txt'
    path.write_text(spec['text'].rstrip() + '\n', encoding='utf-8')
    prompt_paths[name] = path

schema_variants = {}
for count in sorted(set(RELATION_COUNTS)):
    selected_ids = ranked_relation_ids[:count]
    schema = copy.deepcopy(full_schema)
    schema['relations'] = {rid: copy.deepcopy(full_schema['relations'][rid]) for rid in selected_ids}
    schema['built_from']['experiment_relation_count'] = count
    schema['built_from']['selection_rule'] = 'top_train_count_then_relation_id'
    schema_path = EXPERIMENT_ROOT / 'schemas' / f'relations_{count}.json'
    schema_path.write_text(json.dumps(schema, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    covered = sum(relation_counts[rid] for rid in selected_ids)
    schema_variants[count] = {
        'schema_path': str(schema_path), 'relation_ids': selected_ids,
        'train_relation_coverage_share': covered / total_train_facts,
    }

variants = []
repo_paths = base_cfg['paths']
for prompt_name in PROMPT_VARIANTS:
    for relation_count in sorted(set(RELATION_COUNTS)):
        variant_id = f'p-{prompt_name}__r-{relation_count}'
        selected_ids = schema_variants[relation_count]['relation_ids']
        cfg = copy.deepcopy(base_cfg)
        cfg['paths']['train'] = str(REPO_DIR / repo_paths['train'])
        cfg['paths']['dev'] = str(REPO_DIR / repo_paths['dev'])
        cfg['paths']['test'] = str(REPO_DIR / repo_paths['test'])
        cfg['paths']['schema'] = schema_variants[relation_count]['schema_path']
        cfg['paths']['test_freeze'] = str(REPO_DIR / repo_paths['test_freeze'])
        cfg['paths']['raw_dir'] = str(EXPERIMENT_ROOT / 'raw' / variant_id)
        cfg['paths']['results_dir'] = str(EXPERIMENT_ROOT / 'runs' / variant_id)
        cfg['paths']['annotation_dir'] = str(EXPERIMENT_ROOT / 'annotation' / variant_id)
        cfg['paths']['judge_cache'] = str(EXPERIMENT_ROOT / 'judge_cache' / f'{variant_id}.jsonl')
        cfg['extractor']['prompt_file'] = str(prompt_paths[prompt_name])
        cfg['judge']['prompt_file'] = str(REPO_DIR / base_cfg['judge']['prompt_file'])
        cfg['judge']['enabled'] = False
        cfg['schema']['relations'] = [copy.deepcopy(relation_specs[rid]) for rid in selected_ids]
        cfg['schema']['pair_policy'] = PAIR_POLICY
        cfg['eval']['sizes'] = SIZE_PRESETS['all']['eval_sizes']
        cfg['eval']['seeds'] = SIZE_PRESETS['all']['seeds']
        cfg['eval']['primary_n'] = max(cfg['eval']['sizes'])
        cfg['_experiment_variant'] = {
            'variant_id': variant_id, 'prompt_name': prompt_name,
            'relation_count': relation_count, 'pair_policy': PAIR_POLICY,
        }
        config_path = EXPERIMENT_ROOT / 'configs' / f'{variant_id}.yaml'
        config_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
        variants.append({
            'variant_id': variant_id, 'prompt_name': prompt_name,
            'prompt_description': PROMPT_VARIANTS[prompt_name]['description'],
            'prompt_path': str(prompt_paths[prompt_name]),
            'relation_count': relation_count, 'relation_ids': selected_ids,
            'train_relation_coverage_share': schema_variants[relation_count]['train_relation_coverage_share'],
            'entity_types': full_schema['entity_types'], 'pair_policy': PAIR_POLICY,
            'schema_path': schema_variants[relation_count]['schema_path'],
            'config_path': str(config_path),
        })

commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
registry = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'repository_url': REPO_URL, 'repository_commit': commit,
    'baseline_prompt_name': BASELINE_PROMPT_NAME,
    'baseline_relation_count': BASELINE_RELATION_COUNT,
    'full_relation_ids': full_relation_ids, 'ranked_relation_ids': ranked_relation_ids,
    'relation_catalog_size': len(full_relation_ids), 'relation_catalog_path': str(full_catalog_path),
    'relation_metadata_path': str(metadata_path), 'pair_policy': PAIR_POLICY,
    'entity_types_fixed': full_schema['entity_types'],
    'size_presets': SIZE_PRESETS, 'variants': variants,
}
registry_path = EXPERIMENT_ROOT / 'experiment_registry.json'
registry_path.write_text(json.dumps(registry, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'Created {len(variants)} variants from {len(full_relation_ids)} TRAIN relations')
print('Registry:', registry_path)

In [ ]:
import pandas as pd

variant_table = pd.DataFrame([{
    'variant_id': v['variant_id'],
    'prompt_name': v['prompt_name'],
    'relation_count': v['relation_count'],
    'relation_ids': ' '.join(v['relation_ids']),
    'train_relation_coverage_share': v['train_relation_coverage_share'],
    'config_path': v['config_path'],
} for v in variants])
variant_table.to_csv(EXPERIMENT_ROOT / 'experiment_variants.csv', index=False)
display(variant_table)

## Generated artifacts

```text
configuration_search_full96/
├── experiment_registry.json
├── experiment_variants.csv
├── prompts/*.txt
├── schemas/relations_*.json
├── configs/*.yaml
├── raw/
├── runs/
└── comparisons/
```

Next, run `02_colab_experiment_runner.ipynb`. Do not choose the best configuration using precision alone: a smaller schema covers fewer kinds of facts. The runner also computes coverage-adjusted metrics against the shared full schema.